# ELSST Track 1: Prompting Experiments

This notebook covers zero-shot, few-shot, and chain-of-thought prompting for implicit
concept retrieval, using the Gemini API.

## 1. Environment Setup
> Mounting Drive, installing the Gemini SDK, and loading everything this notebook needs
independently dataset, concept pool, evaluation functions since none of that carries
over from the baselines notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

project_path = '/content/drive/MyDrive/ELSST_Project'
RESULTS_DIR = os.path.join(project_path, 'results')
CHECKPOINT_DIR = os.path.join(project_path, 'checkpoints')

Mounted at /content/drive


In [ ]:
!pip install google-genai datasets huggingface_hub -q
print("Libraries installed!")

Libraries installed!


### 1.1 Set up Gemini API key

In [ ]:
from google.colab import userdata
from google import genai

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

MODEL_NAME = "gemini-3.5-flash-lite"

test_response = client.models.generate_content(model=MODEL_NAME, contents="Say hello in one word.")
print(test_response.text)

Hello.


In [ ]:
import time

for i in range(8):
    test_response = client.models.generate_content(model=MODEL_NAME, contents="Say hello in one word.")
    print(f"{i+1}: {test_response.text.strip()}")
    time.sleep(2)

1: Hello.
2: Hello.
3: Hello.
4: Hello.
5: Hello.
6: Hello.
7: Hello.
8: Hello.


### 1.2 Load dataset and concept pool
> Same as the baselines notebook reloading here since this is a separate runtime.

In [ ]:
import json
import numpy as np
from datasets import load_from_disk
from huggingface_hub import hf_hub_download

SEED = 42
np.random.seed(SEED)

dataset = load_from_disk(os.path.join(project_path, 'dataset'))

pool_path = hf_hub_download(
    repo_id="JohnWang10086/elsst-track1",
    filename="concept_pool.jsonl",
    repo_type="dataset"
)

concept_pool_lookup = {}
with open(pool_path, "r") as f:
    for line in f:
        c = json.loads(line)
        concept_pool_lookup[c["concept_id"]] = c

assert len(concept_pool_lookup) == 3433, f"Expected 3433 concepts, got {len(concept_pool_lookup)}"

concept_ids = list(concept_pool_lookup.keys())
val_ids = [ex['id'] for ex in dataset['validation']]
val_texts = [ex['text'] for ex in dataset['validation']]
gold = {ex['id']: ex['retrieval_labels']['positive_ids'] for ex in dataset['validation']}

print(f"Dataset loaded: {len(dataset['train'])} train / {len(dataset['validation'])} val")
print(f"Concept pool loaded: {len(concept_pool_lookup)} concepts")

concept_pool.jsonl:   0%|          | 0.00/800k [00:00<?, ?B/s]

Dataset loaded: 2985 train / 756 val
Concept pool loaded: 3433 concepts


### 1.3 Evaluation functions and caching helper
> Same MRR/Recall@5/Recall@10/NDCG@10 functions from the baselines notebook, so
prompting results stay directly comparable to Section 3's numbers.

In [ ]:
def reciprocal_rank(ranked_ids, positive_ids):
    positive_set = set(positive_ids)
    for rank, cid in enumerate(ranked_ids, start=1):
        if cid in positive_set:
            return 1.0 / rank
    return 0.0

def recall_at_k(ranked_ids, positive_ids, k):
    if not positive_ids:
        return 0.0
    top_k = set(ranked_ids[:k])
    hits = len(top_k & set(positive_ids))
    return hits / len(positive_ids)

def ndcg_at_k(ranked_ids, positive_ids, k=10):
    positive_set = set(positive_ids)
    dcg = sum(
        1.0 / np.log2(i + 1)
        for i, cid in enumerate(ranked_ids[:k], start=1)
        if cid in positive_set
    )
    ideal_hits = min(len(positive_ids), k)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_retrieval(predictions, gold):
    mrr, r5, r10, ndcg = [], [], [], []
    for doc_id, positive_ids in gold.items():
        ranked = predictions[doc_id]
        mrr.append(reciprocal_rank(ranked, positive_ids))
        r5.append(recall_at_k(ranked, positive_ids, 5))
        r10.append(recall_at_k(ranked, positive_ids, 10))
        ndcg.append(ndcg_at_k(ranked, positive_ids, 10))
    return {
        "MRR": float(np.mean(mrr)),
        "Recall@5": float(np.mean(r5)),
        "Recall@10": float(np.mean(r10)),
        "NDCG@10": float(np.mean(ndcg)),
    }


def run_or_load(name, compute_fn):
    path = os.path.join(RESULTS_DIR, f"{name}.json")
    if os.path.exists(path):
        print(f"Loaded cached result: {name}")
        with open(path) as f:
            return json.load(f)
    print(f"Running: {name}")
    result = compute_fn()
    with open(path, "w") as f:
        json.dump(result, f, indent=2)
    return result

### 1.4 Rate-limited LLM call wrapper
> Gemini's free tier caps requests per minute and can occasionally 429 even within
that limit this wrapper spaces calls out and retries once before giving up, so a
single hiccup 300 documents in doesn't crash the whole run.

In [ ]:
import time

def call_llm(prompt, max_retries=3, wait_seconds=4):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
            time.sleep(wait_seconds)
            return response.text
        except Exception as e:
            if "429" in str(e):
                print(f"Rate limited, waiting 10s (attempt {attempt + 1})...")
                time.sleep(10)
            else:
                raise
    raise RuntimeError("Failed after max retries")

## 2. Zero-shot Prompting
> **Approach:** Directly asking Gemini to rank the 3,433-concept pool from scratch isn't
practical even at 20 words per concept, the full pool would run to tens of thousands of
tokens per call, which is both expensive and unnecessary. Instead, this uses a standard
retrieve-then-rerank pattern: take the top-50 candidates from our strongest baseline
Qwen3-Embedding, Section 7 of the baselines notebook and ask the LLM to re-rank just
those, using its language understanding rather than pure embedding similarity.
>
> **Hypothesis:** Since Qwen3's top-50 already contains the correct concept most of the
time Recall@50 should be notably higher than its Recall@10 of 0.362, the LLM's job is
mainly to re-order rather than discover meaning any gains here come from better
reasoning about relevance, not better recall of candidates. Expect a modest MRR/NDCG
improvement over Qwen3 alone, since NDCG rewards exactly this kind of reordering.

In [ ]:
with open(os.path.join(RESULTS_DIR, 'qwen_predictions.json')) as f:
    qwen_predictions = json.load(f)

print(f"Loaded shortlists for {len(qwen_predictions)} documents")
print("Example shortlist size:", len(qwen_predictions[val_ids[0]]))

Loaded shortlists for 756 documents
Example shortlist size: 50


### 2.1 Prompt template
> Presenting the 50 candidates as a numbered list with term + definition, asking for a
JSON array of concept IDs in relevance order JSON keeps the output easy to parse
reliably, rather than trying to regex out a ranked list from free-form prose.

In [ ]:
def build_zero_shot_prompt(doc_text, candidate_ids):
    candidates_block = "\n".join(
        f"{i+1}. [{cid}] {concept_pool_lookup[cid]['term']}: {concept_pool_lookup[cid]['definition']}"
        for i, cid in enumerate(candidate_ids)
    )

    return f"""You are analysing a text to identify which social science concepts it implies,
even when those concepts are never explicitly named. The concepts are drawn from a formal
thesaurus and may be reflected through situations, actions, or themes in the text rather
than stated directly.

TEXT:
{doc_text}

CANDIDATE CONCEPTS:
{candidates_block}

Identify which of these candidate concepts are actually implied by the text, and rank
them from most to least relevant. Respond with ONLY a JSON array of concept IDs in ranked
order, e.g. ["id1", "id2", "id3"]. Include all 50 IDs, just reordered - do not omit any."""

### 2.2 Test run on a handful of documents
> Testing on 5 documents before committing to the full validation set this confirms the JSON parsing actually works reliably and gives a rough sense of output quality before
spending a meaningful chunk of the daily quota on 756 calls.

In [ ]:
import re

def parse_ranked_ids(response_text, valid_ids):
    """Extracts the JSON array from the response and keeps only IDs that were
    actually in the candidate list - guards against the model hallucinating an
    ID that wasn't offered as an option."""
    match = re.search(r'\[.*\]', response_text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON array found in response: {response_text[:200]}")
    ranked = json.loads(match.group())
    valid_set = set(valid_ids)
    return [cid for cid in ranked if cid in valid_set]


test_ids = val_ids[:5]

for doc_id in test_ids:
    idx = val_ids.index(doc_id)
    candidates = qwen_predictions[doc_id]
    prompt = build_zero_shot_prompt(val_texts[idx], candidates)

    response_text = call_llm(prompt)
    ranked = parse_ranked_ids(response_text, candidates)

    print(f"\n--- {doc_id} ---")
    print(f"Gold: {[concept_pool_lookup[cid]['term'] for cid in gold[doc_id]]}")
    print(f"Parsed {len(ranked)}/{len(candidates)} IDs successfully")
    print(f"Top 3 predicted: {[concept_pool_lookup[cid]['term'] for cid in ranked[:3]]}")


--- val_v00029 ---
Gold: ['REGIONAL ECONOMY', 'REGIONAL FINANCE']
Parsed 50/50 IDs successfully
Top 3 predicted: ['REGIONAL FINANCE', 'REGIONAL ECONOMY', 'DECENTRALIZED GOVERNMENT']

--- val_v00010 ---
Gold: ['UNDERGRADUATES']
Parsed 50/50 IDs successfully
Top 3 predicted: ['COST OF LIVING', 'UNDERGRADUATES', 'STUDENTS (COLLEGE)']

--- val_v00002 ---
Gold: ['SINGLE-SEX SCHOOLS', 'RESEARCH CENTRES']
Parsed 49/50 IDs successfully
Top 3 predicted: ['SINGLE-SEX SCHOOLS', 'SCHOOLS', 'EDUCATIONAL RESEARCH']

--- val_v00015 ---
Gold: ['CONSUMER SOCIETIES', 'FUTURE SOCIETY']
Parsed 50/50 IDs successfully
Top 3 predicted: ['PROPERTY, OWNERSHIP AND TENURE', 'POST-MATERIALISM', 'CONSUMER BEHAVIOUR']

--- val_v00035 ---
Gold: ['EMPLOYMENT', 'EMPLOYMENT ABROAD', 'JOB VACANCIES', 'BORDER CONTROLS', 'LAND AND PROPERTY FINANCE']
Parsed 50/50 IDs successfully
Top 3 predicted: ['LABOUR SHORTAGES', 'JOB VACANCIES', 'LABOUR MARKET']


### 2.3 Full validation run

In [ ]:
def compute_zero_shot():
    checkpoint_path = os.path.join(CHECKPOINT_DIR, 'zero_shot_partial.json')
    predictions = {}

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            predictions = json.load(f)
        print(f"Resuming from {len(predictions)}/{len(val_ids)} documents already done")

    for i, doc_id in enumerate(val_ids):
        if doc_id in predictions:
            continue

        idx = val_ids.index(doc_id)
        candidates = qwen_predictions[doc_id]
        prompt = build_zero_shot_prompt(val_texts[idx], candidates)

        try:
            response_text = call_llm(prompt)
            ranked = parse_ranked_ids(response_text, candidates)
            predictions[doc_id] = ranked
        except Exception as e:
            print(f"Failed on {doc_id}: {e}")
            predictions[doc_id] = candidates
        if i % 20 == 0:
            with open(checkpoint_path, 'w') as f:
                json.dump(predictions, f)
            print(f"Checkpoint saved: {len(predictions)}/{len(val_ids)}")

    with open(checkpoint_path, 'w') as f:
        json.dump(predictions, f)

    metrics = evaluate_retrieval(predictions, gold)

    with open(os.path.join(RESULTS_DIR, 'zero_shot_predictions.json'), 'w') as f:
        json.dump(predictions, f)

    return metrics


zero_shot_metrics = run_or_load('zero_shot_prompting', compute_zero_shot)
print(zero_shot_metrics)

Running: zero_shot_prompting
Resuming from 181/756 documents already done
Checkpoint saved: 201/756
Checkpoint saved: 221/756
Checkpoint saved: 241/756
Checkpoint saved: 261/756
Checkpoint saved: 281/756
Checkpoint saved: 301/756
Checkpoint saved: 321/756
Checkpoint saved: 341/756
Checkpoint saved: 361/756
Checkpoint saved: 381/756
Checkpoint saved: 401/756
Checkpoint saved: 421/756
Checkpoint saved: 441/756
Checkpoint saved: 461/756
Checkpoint saved: 481/756
Checkpoint saved: 501/756
Checkpoint saved: 521/756
Checkpoint saved: 541/756
Checkpoint saved: 561/756
Checkpoint saved: 581/756
Checkpoint saved: 601/756
Checkpoint saved: 621/756
Checkpoint saved: 641/756
Checkpoint saved: 661/756
Rate limited, waiting 10s (attempt 1)...
Rate limited, waiting 10s (attempt 2)...
Rate limited, waiting 10s (attempt 3)...
Failed on val_v00673: Failed after max retries
Rate limited, waiting 10s (attempt 1)...
Rate limited, waiting 10s (attempt 2)...
Rate limited, waiting 10s (attempt 3)...
Failed on

In [ ]:
with open(os.path.join(RESULTS_DIR, 'zero_shot_predictions.json')) as f:
    predictions = json.load(f)

failed_ids = [
    doc_id for doc_id in predictions
    if predictions[doc_id] == qwen_predictions[doc_id]
]
print(f"{len(failed_ids)} documents look like fallbacks")

254 documents look like fallbacks


In [ ]:
print(f"Retrying {len(failed_ids)} documents")

still_failed = []
for i, doc_id in enumerate(failed_ids):
    idx = val_ids.index(doc_id)
    candidates = qwen_predictions[doc_id]
    prompt = build_zero_shot_prompt(val_texts[idx], candidates)

    try:
        response_text = call_llm(prompt)
        ranked = parse_ranked_ids(response_text, candidates)
        predictions[doc_id] = ranked
        print(f"[{i+1}/{len(failed_ids)}] Recovered: {doc_id}")
    except Exception as e:
        print(f"[{i+1}/{len(failed_ids)}] Still failing on {doc_id}: {e}")
        still_failed.append(doc_id)

    if i % 20 == 0:
        with open(os.path.join(RESULTS_DIR, 'zero_shot_predictions.json'), 'w') as f:
            json.dump(predictions, f)
        print(f"  (checkpoint saved at {i+1}/{len(failed_ids)})")

print(f"\n{len(failed_ids) - len(still_failed)}/{len(failed_ids)} recovered, {len(still_failed)} still failing")

metrics = evaluate_retrieval(predictions, gold)
with open(os.path.join(RESULTS_DIR, 'zero_shot_predictions.json'), 'w') as f:
    json.dump(predictions, f)
with open(os.path.join(RESULTS_DIR, 'zero_shot_prompting.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print(metrics)

Retrying 254 documents
[1/254] Recovered: val_v00019
  (checkpoint saved at 1/254)
[2/254] Recovered: val_v00037
[3/254] Recovered: val_v00012
[4/254] Recovered: val_v00031
[5/254] Recovered: val_v00026
[6/254] Recovered: val_v00039
[7/254] Recovered: val_v00023
[8/254] Recovered: val_v00036
[9/254] Recovered: val_v00018
[10/254] Recovered: val_v00014
[11/254] Recovered: val_v00022
[12/254] Recovered: val_v00021
[13/254] Recovered: val_v00032
[14/254] Recovered: val_v00007
[15/254] Recovered: val_v00016
[16/254] Recovered: val_v00020
[17/254] Recovered: val_v00008
[18/254] Recovered: val_v00024
[19/254] Recovered: val_v00034
[20/254] Recovered: val_v00009
[21/254] Recovered: val_v00006
  (checkpoint saved at 21/254)
[22/254] Recovered: val_v00017
[23/254] Recovered: val_v00027
[24/254] Recovered: val_v00042
[25/254] Recovered: val_v00013
[26/254] Recovered: val_v00043
[27/254] Recovered: val_v00033
[28/254] Recovered: val_v00063
[29/254] Recovered: val_v00054
[30/254] Recovered: val_v0

## 5. Few-shot Prompting
> **Approach:** Same re-ranking setup as zero-shot Qwen3's top-50 candidates, ranked by
Gemini, but now the prompt includes 2 worked examples pulled directly from train real
passage/concept pairs, not fabricated ones showing the model what a correctly ranked
answer looks like before it tackles the actual document.
>
> **Hypothesis:** Should modestly outperform zero-shot's MRR of 0.501. Worked examples
typically help most with output format consistency and calibrating how many concepts
to rank highly less clear whether it improves the actual semantic reasoning, which is
really what implicit concept detection depends on.

In [ ]:
train_rows = dataset['train']
example_single = next(row for row in train_rows if len(row['generation_labels']) == 1)
example_multi = next(row for row in train_rows if len(row['generation_labels']) >= 4)

few_shot_examples = [example_single, example_multi]

for ex in few_shot_examples:
    concepts = [label['term'] for label in ex['generation_labels']]
    print(f"({len(concepts)} concepts): {concepts}")
    print(ex['text'][:150], "...\n")

(1 concepts): ['INJUNCTIONS']
Walking through the heart of our town on a Tuesday afternoon, you might notice something different. The air feels thinner, the noise levels have dropp ...

(5 concepts): ['AUDIENCE RESEARCH', 'LISTENING', 'AUDIENCES', 'TELEVISION VIEWING', 'WATCHING']
u/MediaMod24: I was looking over the latest quarterly report on the state of media consumption this morning, and it really hit home how much the lands ...



### 5.1 Prompt template
> Truncating each worked example to 300 words full passages would roughly double
prompt length across 756 calls for marginal benefit, since the point is showing the
pattern passage ranked concepts, not the full text.

In [ ]:
def build_few_shot_prompt(doc_text, candidate_ids, examples):
    candidates_block = "\n".join(
        f"{i+1}. [{cid}] {concept_pool_lookup[cid]['term']}: {concept_pool_lookup[cid]['definition']}"
        for i, cid in enumerate(candidate_ids)
    )

    examples_block = ""
    for ex in examples:
        concepts = [label['term'] for label in ex['generation_labels']]
        examples_block += f"\nExample passage: {ex['text'][:300]}...\n"
        examples_block += f"Correct concepts (most to least relevant): {concepts}\n---\n"

    return f"""You are analysing a text to identify which social science concepts it implies,
even when those concepts are never explicitly named. The concepts are drawn from a formal
thesaurus and may be reflected through situations, actions, or themes in the text rather
than stated directly.

Here are some worked examples:
{examples_block}

Now do the same for this new passage:

TEXT:
{doc_text}

CANDIDATE CONCEPTS:
{candidates_block}

Identify which of these candidate concepts are actually implied by the text, and rank
them from most to least relevant. Respond with ONLY a JSON array of concept IDs in ranked
order, e.g. ["id1", "id2", "id3"]. Include all 50 IDs, just reordered - do not omit any."""

### 5.2 Quick test before the full run
> Same precaution as zero-shot confirming parsing works before spending quota on 756 calls.

In [ ]:
test_doc_id = val_ids[0]
test_idx = val_ids.index(test_doc_id)
test_candidates = qwen_predictions[test_doc_id]
test_prompt = build_few_shot_prompt(val_texts[test_idx], test_candidates, few_shot_examples)

response_text = call_llm(test_prompt)
ranked = parse_ranked_ids(response_text, test_candidates)

print(f"Gold: {[concept_pool_lookup[cid]['term'] for cid in gold[test_doc_id]]}")
print(f"Parsed {len(ranked)}/50 IDs")
print(f"Top 3: {[concept_pool_lookup[cid]['term'] for cid in ranked[:3]]}")

Gold: ['REGIONAL ECONOMY', 'REGIONAL FINANCE']
Parsed 50/50 IDs
Top 3: ['REGIONAL ECONOMY', 'REGIONAL FINANCE', 'DECENTRALIZED GOVERNMENT']


### 5.3 Full validation run
> Same checkpointing pattern as zero-shot, but with explicit fallback tracking built in
from the start this time saves re-detecting failures after the fact.

In [ ]:
def compute_few_shot():
    checkpoint_path = os.path.join(CHECKPOINT_DIR, 'few_shot_partial.json')
    failed_path = os.path.join(CHECKPOINT_DIR, 'few_shot_failed_ids.json')
    predictions = {}
    failed_ids = []

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            predictions = json.load(f)
        print(f"Resuming from {len(predictions)}/{len(val_ids)} documents already done")
    if os.path.exists(failed_path):
        with open(failed_path) as f:
            failed_ids = json.load(f)

    for i, doc_id in enumerate(val_ids):
        if doc_id in predictions:
            continue

        idx = val_ids.index(doc_id)
        candidates = qwen_predictions[doc_id]
        prompt = build_few_shot_prompt(val_texts[idx], candidates, few_shot_examples)

        try:
            response_text = call_llm(prompt)
            ranked = parse_ranked_ids(response_text, candidates)
            predictions[doc_id] = ranked
            print(f"[{i+1}/{len(val_ids)}] {doc_id} done")
        except Exception as e:
            print(f"Failed on {doc_id}: {e}")
            predictions[doc_id] = candidates
            failed_ids.append(doc_id)

        if i % 20 == 0:
            with open(checkpoint_path, 'w') as f:
                json.dump(predictions, f)
            with open(failed_path, 'w') as f:
                json.dump(failed_ids, f)
            print(f"Checkpoint saved: {len(predictions)}/{len(val_ids)} ({len(failed_ids)} failed so far)")

    with open(checkpoint_path, 'w') as f:
        json.dump(predictions, f)
    with open(failed_path, 'w') as f:
        json.dump(failed_ids, f)

    metrics = evaluate_retrieval(predictions, gold)
    with open(os.path.join(RESULTS_DIR, 'few_shot_predictions.json'), 'w') as f:
        json.dump(predictions, f)

    print(f"\nFinished. {len(failed_ids)}/{len(val_ids)} documents needed fallback.")
    return metrics

few_shot_metrics = run_or_load('few_shot_prompting', compute_few_shot)
print(few_shot_metrics)

Running: few_shot_prompting
Resuming from 261/756 documents already done
[262/756] val_v00277 done
[263/756] val_v00268 done
[264/756] val_v00256 done
[265/756] val_v00278 done
[266/756] val_v00262 done
[267/756] val_v00253 done
[268/756] val_v00255 done
[269/756] val_v00264 done
[270/756] val_v00252 done
[271/756] val_v00251 done
[272/756] val_v00245 done
[273/756] val_v00258 done
[274/756] val_v00257 done
[275/756] val_v00241 done
[276/756] val_v00269 done
[277/756] val_v00272 done
[278/756] val_v00263 done
[279/756] val_v00260 done
[280/756] val_v00298 done
[281/756] val_v00300 done
Checkpoint saved: 281/756 (28 failed so far)
[282/756] val_v00285 done
[283/756] val_v00311 done
[284/756] val_v00309 done
[285/756] val_v00312 done
[286/756] val_v00292 done
[287/756] val_v00315 done
[288/756] val_v00295 done
[289/756] val_v00307 done
[290/756] val_v00284 done
[291/756] val_v00305 done
[292/756] val_v00299 done
[293/756] val_v00282 done
[294/756] val_v00308 done
[295/756] val_v00310 don

In [ ]:
with open(os.path.join(CHECKPOINT_DIR, 'few_shot_failed_ids.json')) as f:
    failed_ids = json.load(f)

with open(os.path.join(RESULTS_DIR, 'few_shot_predictions.json')) as f:
    predictions = json.load(f)

print(f"Retrying {len(failed_ids)} documents")

still_failed = []
for i, doc_id in enumerate(failed_ids):
    idx = val_ids.index(doc_id)
    candidates = qwen_predictions[doc_id]
    prompt = build_few_shot_prompt(val_texts[idx], candidates, few_shot_examples)

    try:
        response_text = call_llm(prompt)
        ranked = parse_ranked_ids(response_text, candidates)
        predictions[doc_id] = ranked
        print(f"[{i+1}/{len(failed_ids)}] Recovered: {doc_id}")
    except Exception as e:
        print(f"[{i+1}/{len(failed_ids)}] Still failing on {doc_id}: {e}")
        still_failed.append(doc_id)

print(f"\n{len(failed_ids) - len(still_failed)}/{len(failed_ids)} recovered, {len(still_failed)} still failing")

metrics = evaluate_retrieval(predictions, gold)
with open(os.path.join(RESULTS_DIR, 'few_shot_predictions.json'), 'w') as f:
    json.dump(predictions, f)
with open(os.path.join(RESULTS_DIR, 'few_shot_prompting.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print(metrics)

Retrying 37 documents
[1/37] Recovered: val_v00204
[2/37] Recovered: val_v00229
[3/37] Recovered: val_v00205
[4/37] Recovered: val_v00207
[5/37] Recovered: val_v00232
[6/37] Recovered: val_v00206
[7/37] Recovered: val_v00280
[8/37] Recovered: val_v00243
[9/37] Recovered: val_v00254
[10/37] Recovered: val_v00274
[11/37] Recovered: val_v00247
[12/37] Recovered: val_v00249
[13/37] Recovered: val_v00270
[14/37] Recovered: val_v00276
[15/37] Recovered: val_v00244
[16/37] Recovered: val_v00267
[17/37] Recovered: val_v00275
[18/37] Recovered: val_v00271
[19/37] Recovered: val_v00246
[20/37] Recovered: val_v00266
[21/37] Recovered: val_v00242
[22/37] Recovered: val_v00265
[23/37] Recovered: val_v00248
[24/37] Recovered: val_v00250
[25/37] Recovered: val_v00279
[26/37] Recovered: val_v00261
[27/37] Recovered: val_v00273
[28/37] Recovered: val_v00259
[29/37] Recovered: val_v00464
[30/37] Recovered: val_v00724
[31/37] Recovered: val_v00756
[32/37] Recovered: val_v00747
[33/37] Recovered: val_v007

### 5.4 Statistical Significance Testing
> Zero-shot and few-shot produced very close MRR scores 0.5012 vs 0.5059. Before drawing
any conclusion about which is better or whether they're meaningfully different at all this needs a proper significance test, not just a comparison of two averages.
>
> The Wilcoxon signed-rank test checks whether the per document differences between two paired sets of scores are large enough to be unlikely by chance. It's the right test here specifically because both methods were evaluated on the exact same 756 documents this is a paired comparison, not two independent samples.
>
> Reciprocal rank scores aren't normally distributed they're bounded between 0 and 1 and often cluster near 0 or near 1, so a standard paired t-test would be inappropriate. Wilcoxon is the non-parametric equivalent, designed exactly for this kind of skewed, bounded, paired data.
>
> This turns "few-shot scored 0.0047 higher" from an ambiguous number into a
defensible claim either way either the difference is statistically significant or
there is no significant difference, both of which are legitimate findings to report,
unlike guessing from the raw averages alone.

In [ ]:
from scipy.stats import wilcoxon

with open(os.path.join(RESULTS_DIR, 'zero_shot_predictions.json')) as f:
    zero_shot_preds = json.load(f)
with open(os.path.join(RESULTS_DIR, 'few_shot_predictions.json')) as f:
    few_shot_preds = json.load(f)

zero_rr = [reciprocal_rank(zero_shot_preds[doc_id], gold[doc_id]) for doc_id in val_ids]
few_rr = [reciprocal_rank(few_shot_preds[doc_id], gold[doc_id]) for doc_id in val_ids]

stat, p_value = wilcoxon(zero_rr, few_rr)
print(f"Wilcoxon statistic: {stat:.2f}, p-value: {p_value:.4f}")

diffs = np.array(few_rr) - np.array(zero_rr)
print(f"Documents where few-shot did better: {(diffs > 0).sum()}")
print(f"Documents where few-shot did worse: {(diffs < 0).sum()}")
print(f"Documents identical: {(diffs == 0).sum()}")

Wilcoxon statistic: 26187.50, p-value: 0.9350
Documents where few-shot did better: 153
Documents where few-shot did worse: 171
Documents identical: 432


### 5.5 Per-Genre Difference Analysis
> Even if the overall averages are close or the significance test comes back negative,
that could be hiding real effects that cancel out few-shot might genuinely help on some
document types and hurt on others.
>
> Breaking down the per-document score difference few-shot minus zero-shot by `document_type`, to see if the effect is uniform across all 10 genres or concentrated in specific ones.
>
> The dataset's genre balance confirmed in EDA means each of the 10 document types has a similar number of documents, so a genre-level breakdown is a fair, like for like comparison rather than an artefact of some genres being over represented.
>
> If a genre-specific pattern shows up, that's a much more interesting and
specific finding than a flat no difference e.g. it could point to few-shot examples
helping most on document types similar to the worked examples themselves, which would be
a genuine methodological insight worth discussing.

In [ ]:
document_types = {ex['id']: ex['document_type'] for ex in dataset['validation']}

import pandas as pd
diff_df = pd.DataFrame({
    'doc_id': val_ids,
    'doc_type': [document_types[d] for d in val_ids],
    'zero_shot_rr': zero_rr,
    'few_shot_rr': few_rr,
    'diff': diffs
})

print(diff_df.groupby('doc_type')['diff'].mean().sort_values())

doc_type
forum_discussion       -0.041330
case_study             -0.024423
op_ed                  -0.015797
research_summary       -0.013614
policy_brief            0.002077
interview_transcript    0.008834
news_article            0.009102
blog_post               0.015537
encyclopedia_entry      0.049309
report_excerpt          0.057942
Name: diff, dtype: float64


In [ ]:
for ex in few_shot_examples:
    print(f"Example document_type: {ex['document_type']}")

Example document_type: blog_post
Example document_type: forum_discussion


In [ ]:
report_ids = [d for d in val_ids if document_types[d] == 'report_excerpt']
forum_ids = [d for d in val_ids if document_types[d] == 'forum_discussion']

for genre_name, ids in [('report_excerpt', report_ids), ('forum_discussion', forum_ids)]:
    z = [reciprocal_rank(zero_shot_preds[d], gold[d]) for d in ids]
    f = [reciprocal_rank(few_shot_preds[d], gold[d]) for d in ids]
    stat, p = wilcoxon(z, f)
    print(f"{genre_name} (n={len(ids)}): p = {p:.4f}")

report_excerpt (n=75): p = 0.2448
forum_discussion (n=76): p = 0.0222


**Finding:** the per-genre breakdown reveals a real, specific pattern rather than a flat null result. forum_discussion one of the two genres the worked examples themselves belong to blog_post and forum_discussion, confirmed above shows the single largest negative mean difference (-0.041) of all 10 genres, and this is statistically significant on its own Wilcoxon p=0.0222, n=76. That is the opposite of the matching genre helps hypothesis: few-shot performs worse than zero-shot specifically on the genre matching one of its own worked examples, consistent with an anchoring effect rather than a positive transfer effect. blog_post the other matching genre shows a small positive difference (+0.016), not a similar penalty, so the anchoring effect is not uniform across both matching genres. report_excerpt, by contrast, shows the strongest positive effect (+0.058) and is not statistically significant on its own (p=0.245).

**Caveat:** with 10 genres tested, a Bonferroni-corrected significance threshold would be 0.05/10 = 0.005 forum_discussion's p=0.0222 does not survive this correction, so this should be reported as a suggestive, genre-specific pattern worth noting rather than a confirmed effect.

## 4. Chain-of-Thought Prompting
> **Approach:** Same re-ranking task as zero-shot and few-shot Qwen3's top-50 candidates, but now the prompt explicitly asks the model to reason step-by-step identifying the document's themes, setting, and implied issues before producing the final ranking. This tests whether making the reasoning explicit improves judgment quality, rather than relying on the model's implicit reasoning as in zero-shot.
>
> **Hypothesis:** Given that few-shot showed no significant improvement over zero-shot
Wilcoxon p=0.935, the bottleneck for this task may not be output format or pattern
matching it may be the model's semantic reasoning itself. If that's the case, CoT should
show a more meaningful gain than few-shot did, since it directly targets reasoning depth rather than providing surfacelevel examples.

In [ ]:
def build_cot_prompt(doc_text, candidate_ids):
    candidates_block = "\n".join(
        f"{i+1}. [{cid}] {concept_pool_lookup[cid]['term']}: {concept_pool_lookup[cid]['definition']}"
        for i, cid in enumerate(candidate_ids)
    )

    return f"""You are analysing a text to identify which social science concepts it implies,
even when those concepts are never explicitly named. The concepts are drawn from a formal
thesaurus and may be reflected through situations, actions, or themes in the text rather
than stated directly.

Briefly follow these steps (keep each step to 1-2 sentences):
Step 1: Main theme and setting.
Step 2: Implied social, economic, legal, or political issues.
Step 3: Best matching concepts from the list.

Passage:
{doc_text}

Candidate concepts:
{candidates_block}

Then end your response with "FINAL RANKING:" followed by ONLY the ranked concept terms,
one per line, most relevant first."""

In [ ]:
for fname in ['cot_partial.json', 'cot_failed_ids.json']:
    path = os.path.join(CHECKPOINT_DIR, fname)
    if os.path.exists(path):
        os.remove(path)
        print(f"Removed old checkpoint: {fname}")

if os.path.exists(os.path.join(RESULTS_DIR, 'chain_of_thought_prompting.json')):
    os.remove(os.path.join(RESULTS_DIR, 'chain_of_thought_prompting.json'))
    print("Removed old result")

### 4.1 Response parser
> CoT responses mix reasoning text with the final answer, so parsing needs to isolate
everything after "FINAL RANKING:" first unlike zero-shot/few-shot, a plain regex search for a JSON array would risk picking up stray brackets from the reasoning text instead.

In [ ]:
import re

def parse_cot_response(response, candidates):
    term_to_id = {concept_pool_lookup[c]['term'].upper(): c for c in candidates}

    if "FINAL RANKING:" not in response.upper():
        raise ValueError("No 'FINAL RANKING:' marker found - response likely incomplete or malformed")

    marker_index = response.upper().index("FINAL RANKING:")
    ranking_section = response[marker_index + len("FINAL RANKING:"):]
    lines = [line.strip() for line in ranking_section.split("\n") if line.strip()]

    ranked_ids = []
    for line in lines:
        line_no_id = re.sub(r'^\[[0-9a-fA-F-]+\]\s*', '', line)
        line_clean = line_no_id.lstrip("0123456789.- ").strip().upper()
        if line_clean in term_to_id and term_to_id[line_clean] not in ranked_ids:
            ranked_ids.append(term_to_id[line_clean])

    for c in candidates:
        if c not in ranked_ids:
            ranked_ids.append(c)

    return ranked_ids

In [ ]:
ranked = parse_cot_response(response_text, test_candidates)
print(f"Top 5: {[concept_pool_lookup[cid]['term'] for cid in ranked[:5]]}")

Top 5: ['NEIGHBOURHOODS', 'HISTORIC BUILDINGS', 'URBAN RENEWAL', 'COMMUNITY LIFE', 'MODERNIZATION']


### 4.2 Quick test before the full run

In [ ]:
test_doc_id = val_ids[0]
test_idx = val_ids.index(test_doc_id)
test_candidates = qwen_predictions[test_doc_id]
test_prompt = build_cot_prompt(val_texts[test_idx], test_candidates)

response_text = call_llm(test_prompt)
ranked = parse_cot_response(response_text, test_candidates)

print("Raw response")
print(response_text[:500])
print("\n Parsed ")
print(f"Gold: {[concept_pool_lookup[cid]['term'] for cid in gold[test_doc_id]]}")
print(f"Parsed {len(ranked)}/50 IDs")
print(f"Top 3: {[concept_pool_lookup[cid]['term'] for cid in ranked[:3]]}")

Raw response
Step 1: The text examines how specialized industrial sectors and local trade ecosystems within sub-national regions (such as provinces or states) operate independently of national economic trends. It is set within decentralized governance frameworks where local administrations manage their own economic development.

Step 2: Key issues include the autonomy of sub-national regions in managing capital, local taxation, and public spending to support regional industries. It highlights the tension bet

 Parsed 
Gold: ['REGIONAL ECONOMY', 'REGIONAL FINANCE']
Parsed 50/50 IDs
Top 3: ['REGIONAL ECONOMY', 'REGIONAL FINANCE', 'DECENTRALIZED GOVERNMENT']


### 4.3 Full validation run
> Same checkpointing and failure tracking pattern as zero-shot/few-shot.

In [ ]:
def compute_cot():
    checkpoint_path = os.path.join(CHECKPOINT_DIR, 'cot_partial.json')
    failed_path = os.path.join(CHECKPOINT_DIR, 'cot_failed_ids.json')
    predictions = {}
    failed_ids = []

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            predictions = json.load(f)
        print(f"Resuming from {len(predictions)}/{len(val_ids)} documents already done")

    for i, doc_id in enumerate(val_ids):
        if doc_id in predictions:
            continue

        idx = val_ids.index(doc_id)
        candidates = qwen_predictions[doc_id]
        prompt = build_cot_prompt(val_texts[idx], candidates)

        try:
            response_text = call_llm(prompt)
            ranked = parse_cot_response(response_text, candidates)
            predictions[doc_id] = ranked
            print(f"[{i+1}/{len(val_ids)}] {doc_id} done")
        except Exception as e:
            print(f"Failed on {doc_id}: {e}")
            predictions[doc_id] = candidates
            failed_ids.append(doc_id)

        if i % 20 == 0:
            with open(checkpoint_path, 'w') as f:
                json.dump(predictions, f)
            with open(failed_path, 'w') as f:
                json.dump(failed_ids, f)
            print(f"Checkpoint saved: {len(predictions)}/{len(val_ids)} ({len(failed_ids)} failed so far)")

    with open(checkpoint_path, 'w') as f:
        json.dump(predictions, f)
    with open(failed_path, 'w') as f:
        json.dump(failed_ids, f)

    metrics = evaluate_retrieval(predictions, gold)

    with open(os.path.join(RESULTS_DIR, 'cot_predictions.json'), 'w') as f:
        json.dump(predictions, f)

    print(f"\nFinished. {len(failed_ids)}/{len(val_ids)} documents needed fallback.")
    return metrics

cot_metrics = run_or_load('chain_of_thought_prompting', compute_cot)
print(cot_metrics)

Running: chain_of_thought_prompting
[1/756] val_v00029 done
Checkpoint saved: 1/756 (0 failed so far)
[2/756] val_v00010 done
[3/756] val_v00002 done
[4/756] val_v00015 done
[5/756] val_v00035 done
[6/756] val_v00004 done
[7/756] val_v00038 done
[8/756] val_v00001 done
[9/756] val_v00030 done
[10/756] val_v00028 done
[11/756] val_v00003 done
[12/756] val_v00005 done
[13/756] val_v00025 done
[14/756] val_v00040 done
[15/756] val_v00011 done
[16/756] val_v00019 done
[17/756] val_v00037 done
[18/756] val_v00012 done
[19/756] val_v00031 done
[20/756] val_v00026 done
[21/756] val_v00039 done
Checkpoint saved: 21/756 (0 failed so far)
[22/756] val_v00023 done
[23/756] val_v00036 done
[24/756] val_v00018 done
[25/756] val_v00014 done
[26/756] val_v00022 done
[27/756] val_v00021 done
[28/756] val_v00032 done
[29/756] val_v00007 done
[30/756] val_v00016 done
[31/756] val_v00020 done
[32/756] val_v00008 done
[33/756] val_v00024 done
[34/756] val_v00034 done
[35/756] val_v00009 done
[36/756] val_